<a href="https://colab.research.google.com/github/Justinl20/Justinl20-DataScience-GenAI-Submissions/blob/main/7_02_Machine_Learning_Modelling_Student_COMPLETED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning Modelling – Loan Data

**Week 7 Seminar | Option B**

Build and compare classification models using `loan_data.csv`. All features are numeric, with no missing values. The label is `loan_status`. Complete the tasks below in Google Colab.

**Student exercise:** Write your own code in the empty cells.

## Task 1 – Load and Inspect the Dataset

Upload `loan_data.csv` to Colab and load it into a dataframe. Inspect the first few rows, dataset dimensions, column types, and the distribution of `loan_status`. All columns are numeric and there are no missing values; verify these properties.

In [ ]:
from google.colab import files
uploaded = files.upload()
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('loan_data.csv')
print(df.head())
print('Shape:', df.shape)
df.info()
print('Missing values:', df.isna().sum().sum())
print('Target counts:')
print(df['loan_status'].value_counts())

Saving loan_data.csv to loan_data.csv
   age  years_employed  annual_income  credit_score  credit_history_years  \
0   40            17.2          25579           692                   5.3   
1   33             7.3          43087           627                   3.5   
2   42             1.1          20840           689                   8.4   
3   53             0.5          29147           692                   9.8   
4   32            12.5          63657           630                   7.2   

   savings_assets  current_debt  defaults_on_file  delinquencies_last_2yrs  \
0             895         10820                 0                        0   
1             169         16550                 0                        1   
2              17          7852                 0                        0   
3            1480         11603                 0                        1   
4             209         12424                 0                        0   

   derogatory_marks  loan_amou

## Task 2 – Separate Predictors and Target

Set `loan_status` as the target (Y) and use the remaining columns as predictors (X). Check the dimensions of both.

In [ ]:
X = df.drop(columns=['loan_status'])
y = df['loan_status']

print('Predictors (X) shape:', X.shape)
print('Target (y) shape:', y.shape)

Predictors (X) shape: (50000, 15)
Target (y) shape: (50000,)


## Task 3 – Split the Data

Create training and test sets, using 80% of the observations for training and 20% for testing. Preserve the class proportions and use a fixed random seed so your results are reproducible. Check the resulting shapes.

In [ ]:
from sklearn.model_selection import train_test_split

# Split the dataset into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Check and print the dimensions
print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)
print('y_train shape:', y_train.shape)
print('y_test shape:', y_test.shape)

X_train shape: (40000, 15)
X_test shape: (10000, 15)
y_train shape: (40000,)
y_test shape: (10000,)


## Task 4 – Scale the Predictors

Scale the predictors so they are on a comparable scale. Fit the scaling procedure using **training data only**, then apply the same transformation to the test data. Why should the test set not be used to fit the scaler?

In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialize the scaler
scaler = StandardScaler()

# Fit on training data only and transform
X_train_scaled = scaler.fit_transform(X_train)

# Apply the same transformation to the test data
X_test_scaled = scaler.transform(X_test)

# Print statistics to verify scaling (mean should be close to 0, std to 1)
print("Scaled Training Mean (first 5 features):", X_train_scaled.mean(axis=0)[:5].round(3))
print("Scaled Training Std (first 5 features):", X_train_scaled.std(axis=0)[:5].round(3))
print("Scaled Test Mean (first 5 features):", X_test_scaled.mean(axis=0)[:5].round(3))

Scaled Training Mean (first 5 features): [-0.  0. -0. -0. -0.]
Scaled Training Std (first 5 features): [1. 1. 1. 1. 1.]
Scaled Test Mean (first 5 features): [0.017 0.005 0.007 0.013 0.005]


## Task 5 – Train Machine Learning Models

Train and compare **logistic regression**, **random forest**, and **gradient boosting decision trees (GBDT)** with reasonable initial settings. Use the scaled predictors for logistic regression. Tree-based models do not require scaling, so you can use the unscaled predictors for them.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# 1. Initialize models
log_reg = LogisticRegression(random_state=42, max_iter=1000)
rf_clf = RandomForestClassifier(random_state=42, n_estimators=100, max_depth=10)
gb_clf = GradientBoostingClassifier(random_state=42, n_estimators=100, learning_rate=0.1)

# 2. Train the models
print("Training Logistic Regression...")
log_reg.fit(X_train_scaled, y_train)

print("Training Random Forest...")
rf_clf.fit(X_train, y_train)

print("Training Gradient Boosting Decision Trees...")
gb_clf.fit(X_train, y_train)

print("All models trained successfully!")

Training Logistic Regression...
Training Random Forest...
Training Gradient Boosting Decision Trees...
All models trained successfully!


## Task 6 – Evaluate and Compare Models

Evaluate each model on **both the training and test sets**. Report accuracy, macro precision, macro recall, and macro F1-score. Compare training and test performance. Which model generalizes best, and is there evidence of overfitting?

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Helper function to compute classification metrics
def get_metrics(y_true, y_pred):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Macro Precision': precision_score(y_true, y_pred, average='macro'),
        'Macro Recall': recall_score(y_true, y_pred, average='macro'),
        'Macro F1': f1_score(y_true, y_pred, average='macro')
    }

# 1. Predictions
# Logistic Regression (requires scaled data)
pred_train_lr = log_reg.predict(X_train_scaled)
pred_test_lr = log_reg.predict(X_test_scaled)

# Random Forest
pred_train_rf = rf_clf.predict(X_train)
pred_test_rf = rf_clf.predict(X_test)

# GBDT
pred_train_gb = gb_clf.predict(X_train)
pred_test_gb = gb_clf.predict(X_test)

# 2. Compile metrics
results = {
    'Logistic Regression (Train)': get_metrics(y_train, pred_train_lr),
    'Logistic Regression (Test)': get_metrics(y_test, pred_test_lr),
    'Random Forest (Train)': get_metrics(y_train, pred_train_rf),
    'Random Forest (Test)': get_metrics(y_test, pred_test_rf),
    'Gradient Boosting (Train)': get_metrics(y_train, pred_train_gb),
    'Gradient Boosting (Test)': get_metrics(y_test, pred_test_gb),
}

# 3. Present as a DataFrame
results_df = pd.DataFrame(results).T
display(results_df.round(4))

,Accuracy,Macro Precision,Macro Recall,Macro F1
Logistic Regression (Train),0.8390,0.8385,0.8356,0.8367
Logistic Regression (Test),0.8374,0.8363,0.8345,0.8353
Random Forest (Train),0.9032,0.9048,0.8996,0.9016
Random Forest (Test),0.8819,0.8829,0.8782,0.8800
Gradient Boosting (Train),0.8920,0.8923,0.8893,0.8905
Gradient Boosting (Test),0.8870,0.8874,0.8839,0.8854


## Reflection

Which model would you choose based on the test results? What other checks would you want before using the model to support real lending decisions?


*   Logistic Regression:

Train F1: 0.8367 | Test F1: 0.8353
Overfitting: Minimal. The gap between training and testing performance is extremely small (0.14%), indicating excellent generalization, though overall accuracy (83.7%) is lower than the tree models.

*   Random Forest:

Train F1: 0.9016 | Test F1: 0.8800
Overfitting: Present. There is a drop of ~2.16% in F1-score from train to test, showing that the model is slightly over-relying on patterns specific to the training set.

*   GBDT:

Train F1: 0.8905 | Test F1: 0.8854
Overfitting: Very low. The performance drop from train to test is only ~0.51% while maintaining an outstanding Test F1-score of 88.54% (and 88.7% Test Accuracy).


**Conclusion**
GBDT is best overall choice. It offers best balance where it significantly outperforms Logistic regression and achieves higher test metrics than random forest with minimal overfitting

### Detailed Checks Needed Before Deploying a Lending Model

#### 1. Fairness and Bias Checks
* Making sure the computer doesn't unfairly reject people based on things like their age, gender, race, or where they live. If we aren't careful, the AI can copy past human prejudices hidden in old data. If the model unfairly discriminates, it violates the law, which can lead to massive lawsuits, heavy regulatory fines, and damage to the bank's reputation.

#### 2. Stress Testing and Robustness
* Testing how the model handles bad economic situations (like a severe recession, rising inflation). Models are usually trained on "normal" or "good" historical data. If the economy crashes, many people might lose their jobs or see their incomes drop. We need to simulate these extreme conditions to ensure the model won't make bad decisions that could lead to massive loan defaults and bankrupt the bank.

#### 3. Feature Importance and Explainability
* Being able to explain exactly why the AI decided to approve or reject a specific person's loan request. AI models like GBDTs are "black boxes". However, legally, if a lender rejects a borrower, they must give them a clear, honest reason (an "Adverse Action Notice"). Having explanations (using tools like SHAP) ensures we can provide these legally required reasons and help human loan officers double-check that the AI is using logical reasoning.